# 01 — Data Exploration

Goal: get a feel for what a TESS light curve looks like, and what a real exoplanet transit looks like inside one. We use a confirmed system, **TOI-700** (TIC 150428135), which hosts at least three planets.

**What to look for:**
1. The raw light curve has slow stellar variability (starspots) and gaps where the spacecraft was downlinking data.
2. After flattening, the slow trend is gone but the transits remain.
3. After phase-folding on the planet's known period, all transits stack on top of each other and the dip becomes obvious.

Tip: `lightkurve` caches downloads in `data/raw/.lightkurve/` so re-runs are fast.

In [ ]:
import lightkurve as lk
import matplotlib.pyplot as plt

from exoplanet_hunter.preprocess import build_views, clean_lightcurve, flatten_lightcurve

TIC = 150428135   # TOI-700

In [ ]:
search = lk.search_lightcurve(f"TIC {TIC}", mission="TESS", author="SPOC")
search

In [ ]:
# Download a single sector for speed (use download_all().stitch() for the full mission)
lc = search[0].download()
lc.plot();

In [ ]:
# Clean + flatten
lc_clean = clean_lightcurve(lc, sigma_clip=5.0, min_points=100)
lc_flat  = flatten_lightcurve(lc_clean, window_length=301)
lc_flat.plot();

In [ ]:
# Phase-fold on the known period of TOI-700 b
PERIOD_b   = 9.977       # days
T0_b       = 1493.62     # BJD - 2457000
DURATION_b = 1.6 / 24.0  # days

views = build_views(lc_flat, period=PERIOD_b, t0=T0_b, duration=DURATION_b)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(views.global_view); axes[0].set_title("global view (full phase)")
axes[1].plot(views.local_view);  axes[1].set_title("local view (±3 durations)")
for ax in axes:
    ax.set_xlabel("bin"); ax.set_ylabel("normalised flux")
plt.tight_layout();

**Try yourself:**
- Replace `TIC` with another TOI (e.g. 261136679 = TOI-270).
- Phase-fold on a *wrong* period — the dip washes out. This is why getting the period right matters.
- Increase `sigma_clip` from 5 to 3 — watch how the transit may start being clipped if too aggressive.